In [11]:
import pandas as pd

sample_parts = []

for chunk in pd.read_csv(
    r"C:\data\complaints.csv",
    chunksize=100000,
    dtype=str,
    on_bad_lines="skip"
):
    chunk = chunk[
        chunk["Consumer complaint narrative"].notna()
    ]

    # Take 1% from each chunk
    sample_parts.append(
        chunk.sample(
            frac=0.01,
            random_state=42
        )
    )

sample_df = pd.concat(sample_parts, ignore_index=True)

In [13]:
print(sample_df.shape)

(29807, 18)


In [14]:
print(sample_df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [5]:
df_clean = pd.read_csv(
    r"C:\data\cleaned_complaints.csv",
    dtype=str
)


In [9]:
print(sampled_df.columns.tolist())

['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'clean_narrative']


In [6]:
product_counts = df_clean["Product"].value_counts()

print(product_counts)
print("\nPercentages:")
print((product_counts / len(df_clean) * 100).round(2))

Product
Checking or savings account                                140319
Credit card or prepaid card                                108667
Money transfer, virtual currency, or money service          97188
Credit card                                                 80667
Payday loan, title loan, or personal loan                   17238
Payday loan, title loan, personal loan, or advance loan      8896
Money transfers                                              1497
Name: count, dtype: int64

Percentages:
Product
Checking or savings account                                30.88
Credit card or prepaid card                                23.91
Money transfer, virtual currency, or money service         21.38
Credit card                                                17.75
Payday loan, title loan, or personal loan                   3.79
Payday loan, title loan, personal loan, or advance loan     1.96
Money transfers                                             0.33
Name: count, dtype: float64

In [15]:
sample_size = 12000

sampled_df = (
    df_clean
    .groupby("Product", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=max(1, round(len(x) / len(df_clean) * sample_size)),
            random_state=42
        )
    )
)

print(sampled_df.shape)

(12000, 18)


In [21]:
import pandas as pd

df = pd.read_csv(
    r"C:\data\complaints.csv",
    usecols=[
        "Complaint ID",
        "Product",
        "Consumer complaint narrative"
    ],
    dtype=str,
    nrows=200000   # start with a manageable sample
)

In [22]:
print(df.columns.tolist())
print(sampled_df.columns.tolist())

['Product', 'Consumer complaint narrative', 'Complaint ID']
['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'clean_narrative']


In [23]:
clean_df = df[
    [
        "Complaint ID",
        "Product",
        "Consumer complaint narrative"
    ]
].copy()

In [24]:
print(clean_df["Product"].value_counts())

Product
Credit reporting or other personal consumer reports                             178921
Debt collection                                                                  10986
Credit card                                                                       3281
Checking or savings account                                                       2446
Money transfer, virtual currency, or money service                                1117
Mortgage                                                                           865
Vehicle loan or lease                                                              813
Student loan                                                                       716
Payday loan, title loan, personal loan, or advance loan                            483
Prepaid card                                                                       206
Debt or credit management                                                          126
Credit reporting, credit repair ser

In [26]:
print("clean_df columns")
print(clean_df.columns.tolist())

clean_df columns
['Complaint ID', 'Product', 'Consumer complaint narrative']


In [27]:
print("sampled_df columns")
print(sampled_df.columns.tolist())

sampled_df columns
['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID', 'clean_narrative']


In [28]:
target_products = [
    "Credit card",
    "Checking or savings account",
    "Money transfer, virtual currency, or money service",
    "Payday loan, title loan, personal loan, or advance loan"
]

filtered_df = clean_df[
    clean_df["Product"].isin(target_products)
].copy()

In [29]:
filtered_df["Product"].value_counts()

Product
Credit card                                                3281
Checking or savings account                                2446
Money transfer, virtual currency, or money service         1117
Payday loan, title loan, personal loan, or advance loan     483
Name: count, dtype: int64

In [37]:
print(len(filtered_df))

7327


In [38]:
sampled_df = filtered_df.copy()

print(sampled_df.shape)

(7327, 3)


In [40]:
sample_size = 5000

from sklearn.model_selection import train_test_split

sampled_df, _ = train_test_split(
    filtered_df,
    train_size=sample_size,
    stratify=filtered_df["Product"],
    random_state=42
)

In [41]:
original_dist = (
    df_clean["Product"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

sample_dist = (
    sampled_df["Product"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

comparison = pd.DataFrame({
    "Original %": original_dist,
    "Sample %": sample_dist
})

print(comparison)

                                                    Original %  Sample %
Product                                                                 
Checking or savings account                              30.88     33.38
Credit card                                              17.75     44.78
Credit card or prepaid card                              23.91       NaN
Money transfer, virtual currency, or money service       21.38     15.24
Money transfers                                           0.33       NaN
Payday loan, title loan, or personal loan                 3.79       NaN
Payday loan, title loan, personal loan, or adva...        1.96      6.60


In [42]:
sampled_df.to_csv(
    "cfpb_stratified_sample.csv",
    index=False
)

In [48]:
import langchain
print(langchain.__version__)

1.3.11


In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [51]:
import re

def clean_text(text):
    text = str(text).lower()

    text = re.sub(
        r"i am writing to file a complaint|i would like to file a complaint",
        "",
        text
    )

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

sampled_df["clean_narrative"] = (
    sampled_df["Consumer complaint narrative"]
    .fillna("")
    .apply(clean_text)
)

In [52]:
print(sampled_df.columns.tolist())

['Complaint ID', 'Product', 'Consumer complaint narrative', 'clean_narrative']


In [53]:
chunks = []

for idx, row in sampled_df.iterrows():

    text = row["clean_narrative"]

    split_texts = text_splitter.split_text(text)

    for chunk_id, chunk in enumerate(split_texts):

        chunks.append({
            "complaint_id": row["Complaint ID"],
            "product": row["Product"],
            "chunk_id": chunk_id,
            "text": chunk
        })

In [54]:
import pandas as pd

chunks_df = pd.DataFrame(chunks)

print(chunks_df.shape)
chunks_df.head()

(2098, 4)


,complaint_id,product,chunk_id,text
0,14060912,Credit card,0,we called citi bank customer service on xx xx ...
1,14060912,Credit card,1,manager did not disclose to us what was going ...
2,13851611,Credit card,0,i am disputing xxxx returned check fees charge...
3,13851611,Credit card,1,my checking account despite this a returned ch...
4,13851611,Credit card,2,may violate consumer protection regulations i ...


In [55]:
chunks_df["chunk_length"] = (
    chunks_df["text"]
    .str.split()
    .str.len()
)

print(chunks_df["chunk_length"].describe())

count    2098.000000
mean       80.069113
std        22.386075
min        10.000000
25%        75.250000
50%        88.000000
75%        95.000000
max       114.000000
Name: chunk_length, dtype: float64


In [56]:
chunk_counts = (
    chunks_df.groupby("complaint_id")
    .size()
)

print(chunk_counts.describe())

count    640.000000
mean       3.278125
std        2.249846
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       19.000000
dtype: float64


In [58]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Soret\rag-complaint-chatbo\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Soret\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [59]:
sample_text = chunks_df["text"].iloc[0]

embedding = embedding_model.encode(sample_text)

print(embedding.shape)

(384,)


In [60]:
chunks_df["embedding"] = chunks_df["text"].apply(
    lambda x: embedding_model.encode(x)
)

In [61]:
embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

chunks_df["embedding"] = list(embeddings)

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [63]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [64]:
embeddings = model.encode(
    chunks_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [65]:
print(embeddings.shape)

(2098, 384)


In [66]:
import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 2098


In [71]:
metadata = chunks_df[
    [
        "complaint_id",
        "product",
        "chunk_id",
        "text"
    ]
].to_dict("records")

In [72]:
faiss.write_index(
    index,
    "cfpb_faiss.index"
)

In [73]:
import pickle

with open(
    "cfpb_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)

In [74]:
query = "unauthorized credit card charge"

query_embedding = model.encode(
    [query]
).astype("float32")

In [75]:
distances, indices = index.search(
    query_embedding,
    k=5
)

In [78]:
for idx in indices[0]:
    print(metadata[idx]["complaint_id"])
    print(metadata[idx]["product"])
    print(metadata[idx]["text"][:300])
    print("-" * 50)

13696620
Credit card
were cancelled the credit card company is acting in fraudulent matter making me pay a charge for something i no longer have
--------------------------------------------------
13859509
Credit card
wont do anything about it and are charging me crazy fees for not paying the balance that the thief left on the card its punitive and against my rights as a consumer i do not have the card number as it was stolen from me
--------------------------------------------------
13343696
Credit card
i took out a store card with synchrony bank at a xxxx xxxx retail store at the time of purchase the xxxx xxxx store associate took my information and qualified me for the card signing me up for an optional payment card security program without my permission or knowledge this was not identified until
--------------------------------------------------
13769910
Credit card
and make these charges i have been paying back monies that i did not spend and they xxxx my xxxx by xxxx my account i 

In [80]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="cfpb_complaints"
)

In [ ]:
for i, row in chunks_df.iterrows():

    collection.add(
        ids=[str(i)],
        documents=[row["text"]],
        metadatas=[
            {
                "complaint_id": str(row["complaint_id"]),
                "product": row["product"],
                "chunk_id": int(row["chunk_id"])
            }
        ]
    )

C:\Users\Soret\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:19<00:00, 4.19MiB/s]


In [ ]:
results = collection.query(
    query_texts=[
        "unauthorized credit card charge"
    ],
    n_results=5
)

print(results)